In [2]:
!pip -q install -U anthropic datasets pandas tqdm tenacity scikit-learn

In [3]:
import os
from getpass import getpass

os.environ["ANTHROPIC_API_KEY"] = getpass("Enter your Anthropic API key: ")
print("Anthropic API key loaded securely.")

Enter your Anthropic API key: ··········
Anthropic API key loaded securely.


In [4]:
import os
import re
import json
import time
import pandas as pd

from json import JSONDecodeError
from tqdm import tqdm
from datasets import load_dataset

from anthropic import (
    Anthropic,
    RateLimitError,
    APIConnectionError,
    BadRequestError,
    AuthenticationError
)

from tenacity import (
    retry,
    wait_exponential,
    stop_after_attempt,
    retry_if_exception_type
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

MODEL = "claude-opus-4-7"
DATASET_NAME = "ChanceFocus/flare-causal20-sc"

# First test with 100 samples.
# Set LIMIT = None for the full test set.
# Full test set has around 8.6k rows, so 100 is safer and cheaper.
LIMIT = 100

dataset = load_dataset(DATASET_NAME)

split_name = "test" if "test" in dataset else list(dataset.keys())[0]
split_data = dataset[split_name]

print(dataset)
print("Split:", split_name)
print("Columns:", split_data.column_names)
print("Example:", split_data[0])

VALID_LABELS = ["noise", "causal"]

SYSTEM_PROMPT = """
You are a financial text classification model.

Task:
Classify the input sentence as either "causal" or "noise".

Definitions:
- causal: the sentence indicates a causal relationship between financial events, conditions, decisions, or outcomes.
- noise: the sentence does not indicate a clear causal relationship.

Rules:
1. Return exactly one label: "causal" or "noise".
2. Do not explain your answer.
3. Return only valid JSON in this exact format:
{"label": "noise"}
"""

def normalize_label(x):
    x = str(x).strip().lower()

    if x in ["0", "noise"]:
        return "noise"

    if x in ["1", "causal"]:
        return "causal"

    return "noise"

def get_gold_label(row):
    if "answer" in row and row["answer"] is not None:
        return normalize_label(row["answer"])

    if "gold" in row and row["gold"] is not None:
        return normalize_label(row["gold"])

    if "label" in row and row["label"] is not None:
        return normalize_label(row["label"])

    raise ValueError("Cannot find gold label from row.")

def get_text(row):
    if "text" in row and row["text"] is not None:
        return row["text"]

    if "query" in row and row["query"] is not None:
        return row["query"]

    raise ValueError("Cannot find text from row.")

def parse_claude_json(raw_text):
    raw_text = str(raw_text).strip()

    if raw_text == "":
        raise ValueError("Empty model output")

    try:
        return json.loads(raw_text)

    except JSONDecodeError:
        match = re.search(r"\{.*\}", raw_text, flags=re.DOTALL)

        if match:
            return json.loads(match.group(0))

        print("Raw model output:")
        print(repr(raw_text))
        raise

@retry(
    retry=retry_if_exception_type((RateLimitError, APIConnectionError, JSONDecodeError, ValueError)),
    wait=wait_exponential(multiplier=5, min=10, max=120),
    stop=stop_after_attempt(5),
    reraise=True
)
def call_claude_causal20(text):
    user_prompt = (
        f"Text:\n{text}\n\n"
        'Return JSON only, for example: {"label": "causal"}'
    )

    try:
        message = client.messages.create(
            model=MODEL,
            max_tokens=200,
            system=SYSTEM_PROMPT,
            messages=[
                {
                    "role": "user",
                    "content": user_prompt
                }
            ]
        )

    except BadRequestError as e:
        print("\nBadRequestError detail:")
        print(e)
        raise

    except AuthenticationError as e:
        print("\nAuthenticationError detail:")
        print(e)
        raise

    raw_output = "".join(
        block.text for block in message.content
        if getattr(block, "type", None) == "text"
    )

    parsed = parse_claude_json(raw_output)

    if "label" not in parsed:
        raise ValueError(f"Missing label field in model output: {parsed}")

    return parsed

sample_size = len(split_data) if LIMIT is None else min(LIMIT, len(split_data))
eval_data = split_data.select(range(sample_size))

rows = []
gold_labels = []
pred_labels = []

for idx, row in enumerate(tqdm(eval_data)):
    text = get_text(row)
    gold = get_gold_label(row)

    try:
        result = call_claude_causal20(text)
        pred = normalize_label(result.get("label", "noise"))
        error = ""

    except BadRequestError as e:
        print(f"\nStopping at row {idx} because of BadRequestError.")
        print(e)
        raise

    except AuthenticationError as e:
        print(f"\nStopping at row {idx} because of AuthenticationError.")
        print(e)
        raise

    except Exception as e:
        pred = "noise"
        error = str(e)
        print(f"Error at row {idx}: {e}")

    gold_labels.append(gold)
    pred_labels.append(pred)

    rows.append({
        "idx": idx,
        "id": row.get("id", ""),
        "text": text,
        "gold_label": gold,
        "predicted_label": pred,
        "correct": gold == pred,
        "error": error
    })

    # Slow down to reduce rate-limit risk.
    time.sleep(5)

accuracy = accuracy_score(gold_labels, pred_labels)

macro_precision = precision_score(
    gold_labels,
    pred_labels,
    labels=VALID_LABELS,
    average="macro",
    zero_division=0
)

macro_recall = recall_score(
    gold_labels,
    pred_labels,
    labels=VALID_LABELS,
    average="macro",
    zero_division=0
)

macro_f1 = f1_score(
    gold_labels,
    pred_labels,
    labels=VALID_LABELS,
    average="macro",
    zero_division=0
)

weighted_f1 = f1_score(
    gold_labels,
    pred_labels,
    labels=VALID_LABELS,
    average="weighted",
    zero_division=0
)

print("\n===== Claude Opus 4.7 FLARE-Causal20-SC Evaluation =====")
print(f"Dataset: {DATASET_NAME}")
print(f"Split: {split_name}")
print(f"Model: {MODEL}")
print(f"Samples evaluated: {len(eval_data)}")
print(f"Accuracy: {accuracy:.4f}")
print(f"Macro Precision: {macro_precision:.4f}")
print(f"Macro Recall: {macro_recall:.4f}")
print(f"Macro F1 Score: {macro_f1:.4f}")
print(f"Weighted F1 Score: {weighted_f1:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        gold_labels,
        pred_labels,
        labels=VALID_LABELS,
        zero_division=0
    )
)

print("\nConfusion Matrix:")
confusion_df = pd.DataFrame(
    confusion_matrix(gold_labels, pred_labels, labels=VALID_LABELS),
    index=[f"gold_{x}" for x in VALID_LABELS],
    columns=[f"pred_{x}" for x in VALID_LABELS]
)
print(confusion_df)

df = pd.DataFrame(rows)

df.to_csv("Claude_Opus_4_7_FLARE_Causal20_SC_Evaluation_Results.csv", index=False)

metrics = {
    "dataset": DATASET_NAME,
    "split": split_name,
    "model": MODEL,
    "samples_evaluated": len(eval_data),
    "accuracy": accuracy,
    "macro_precision": macro_precision,
    "macro_recall": macro_recall,
    "macro_f1_score": macro_f1,
    "weighted_f1_score": weighted_f1
}

with open("Claude_Opus_4_7_FLARE_Causal20_SC_Metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

num_errors = df["error"].astype(str).str.len().gt(0).sum()

print("\n===== Error Summary =====")
print(f"Failed samples: {num_errors}")
print(f"Total samples: {len(df)}")
print(f"Failure rate: {num_errors / len(df):.4f}")

pd.set_option("display.max_colwidth", 120)
df.head()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/529 [00:00<?, ?B/s]

data/test-00000-of-00001-daf06f7fa9fb3bf(…):   0%|          | 0.00/2.50M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/8628 [00:00<?, ? examples/s]

DatasetDict({
    test: Dataset({
        features: ['id', 'query', 'answer', 'text', 'choices', 'gold'],
        num_rows: 8628
    })
})
Split: test
Columns: ['id', 'query', 'answer', 'text', 'choices', 'gold']
Example: {'id': 'causal20sc0', 'query': "In this task, you are provided with sentences extracted from financial news and SEC data. Your goal is to classify each sentence into either 'causal' or 'noise' based on whether or not it indicates a causal relationship between financial events. Please return only the category 'causal' or 'noise'.\nText:  Third Democratic presidential debate  September 12, 2019 at 9:54 PM EDT - Updated September 13 at 12:54 AM  WASHINGTON (AP)  -  Ten Democrats seeking the presidency tripped over some details Thursday night as they sparred in a debate thick with policy and personal stories. Several made provocative accusations that President Donald Trump inspired the deadly shooting in El Paso, Texas, last month.\nAnswer:", 'answer': 'noise', 'text': ' 

100%|██████████| 100/100 [10:59<00:00,  6.60s/it]


===== Claude Opus 4.7 FLARE-Causal20-SC Evaluation =====
Dataset: ChanceFocus/flare-causal20-sc
Split: test
Model: claude-opus-4-7
Samples evaluated: 100
Accuracy: 0.7300
Macro Precision: 0.5179
Macro Recall: 0.8636
Macro F1 Score: 0.4555
Weighted F1 Score: 0.8344

Classification Report:
              precision    recall  f1-score   support

       noise       1.00      0.73      0.84        99
      causal       0.04      1.00      0.07         1

    accuracy                           0.73       100
   macro avg       0.52      0.86      0.46       100
weighted avg       0.99      0.73      0.83       100


Confusion Matrix:
             pred_noise  pred_causal
gold_noise           72           27
gold_causal           0            1

===== Error Summary =====
Failed samples: 0
Total samples: 100
Failure rate: 0.0000


,idx,id,text,gold_label,predicted_label,correct,error
0,0,causal20sc0,"Third Democratic presidential debate September 12, 2019 at 9:54 PM EDT - Updated September 13 at 12:54 AM WASHING...",noise,causal,False,
1,1,causal20sc1,"On the policy front, Bernie Sanders claimed his approach to health care has a stamp of approval from everyone who s...",noise,noise,True,
2,2,causal20sc2,Joe Biden misrepresented recent history when he said the administration he served as vice president didn't put migr...,noise,noise,True,
3,3,causal20sc3,"Here's a look at some of the assertions in the third round of Democratic primary debates, the first to have all qua...",noise,causal,False,
4,4,causal20sc4,"It killed 22 people, and injured many more, we were not defeated by that. Nor were we defined by that.",noise,noise,True,
